# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omaradelahmed/fly-rank-internship1/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

We combine the Week-4 rule reason codes with the Week-5 Random Forest score into **one of four
standard review actions** (the same four used by FlyRank's own review-queue playbook):

| Action | When it fires | Why |
|---|---|---|
| **Verify then Review** | Second-half impressions are 7x+ the first-half, with enough volume to trust the ratio (first-half >= 50 impressions) | Big jumps are often a tracking artifact or an ad campaign, not organic growth -- verify before trusting it |
| **Investigate Quiet Risk** | Model risk score >= 0.5 AND no baseline rule fired | The highest-value case for a human: the model sees something the hand-written rules miss entirely |
| **Review before Revert** | Any baseline rule fired (`any_rule_triggered == 1`) | A rule gives a concrete, human-readable reason -- check the hypothesis before editing |
| **Monitor Only** | Everything else: no rule fired, no spike, model risk is low | Healthy, steady content -- watch it, don't touch it |

Rules give the reviewer the *reason*; the model gives the *order*.

### Archetype -> action mapping

Each baseline rule corresponds to a recognizable content archetype. Mapping archetype to a
tactical action (not just a review category) makes the queue easier to act on at a glance:

| Archetype (rule) | Typical profile | Tactical action |
|---|---|---|
| `stale_visible_page` | High traffic, update date unknown/old | Refresh: update facts, dates, examples |
| `declining_with_demand` | Real demand, losing impressions within the window | Investigate cause, consider refresh or consolidation check |
| `thin_content` | Above-median traffic, bottom-quartile word count | Expand: add missing subtopics, depth |
| `page_one_decay_risk` | Ranking well, old content | Protect: refresh before it drifts out of position |
| *(no rule; high model score)* | "Investigate Quiet Risk" | No template action -- requires human diagnosis first |

### The decay/refresh insight

`content_age_days` is the single most important feature to the Random Forest (see Week-5
feature importance) -- consistent with the general pattern that older, unrefreshed content is
disproportionately likely to be flagged as declining. The cell below quantifies this directly
on our own data.

In [ ]:
import numpy as np

# Score every page with the Random Forest trained in w05 (X_encoded already excludes
# imp_label_window and every other label-derived column)
df['rf_proba'] = rf_model.predict_proba(X_encoded)[:, 1]

def assign_action(row):
    # Only trust the spike ratio when there's enough volume behind it
    if row['imp_first_half'] >= 50:
        ratio = row['imp_second_half'] / row['imp_first_half']
    else:
        ratio = 0

    if ratio >= 7:
        return 'Verify then Review'
    if row['rf_proba'] >= 0.5 and row['any_rule_triggered'] == 0:
        return 'Investigate Quiet Risk'
    if row['any_rule_triggered'] == 1:
        return 'Review before Revert'
    return 'Monitor Only'

df['action'] = df.apply(assign_action, axis=1)

print(df['action'].value_counts())
print((df['action'].value_counts(normalize=True) * 100).round(1))

action_queue = df.sort_values('rf_proba', ascending=False)[
    ['rank', 'content_hash_id', 'client_hash_id', 'rf_proba', 'baseline_score',
     'reason_codes', 'action', 'is_declining']
].reset_index(drop=True)

action_queue.head(20)

In [ ]:
rule_cols = ['stale_visible_page', 'declining_with_demand', 'thin_content', 'page_one_decay_risk']

print("How often each archetype rule co-occurs with each action (row %):")
for rule in rule_cols:
    print(f"\n{rule}:")
    print((df.loc[df[rule] == 1, 'action'].value_counts(normalize=True) * 100).round(1))

In [ ]:
age_bins = [0, 90, 180, 270, 365, 10000]
age_labels = ['0-90d', '91-180d', '181-270d', '271-365d', '365d+']
df['age_bucket'] = pd.cut(df['content_age_days'], bins=age_bins, labels=age_labels)

decay_table = df.groupby('age_bucket', observed=True).agg(
    n_pages=('content_hash_id', 'count'),
    decline_rate=('is_declining', 'mean'),
    stale_rule_rate=('stale_visible_page', 'mean'),
).round(3)

print(decay_table)
print("\nTakeaway: if decline_rate climbs with age, it supports refreshing older content before")
print("it drifts further -- the same insight FlyRank's own paper reports (health peaks ~61-90d,")
print("decays toward 271-365d).")

## 2. Intended use and limits

*Who uses this, for what -- and where it stops being valid.*

**Who uses it:** a content review team with limited weekly capacity (e.g. 20-100 pages), using
this queue to decide which pages to look at first -- not an automated publishing or editing
pipeline.

**What it validly supports:** prioritization. "Which page should a human look at first?" is a
ranking problem, and that's exactly what's been validated here (client-holdout split, honest
precision@K against a leakage-audited baseline).

**Cost/value thinking:** a reviewer's time is the scarce resource. A false positive here costs
one wasted review (a few minutes); a false negative costs a real decline going unreviewed for a
month. Given that asymmetry, the queue is deliberately tuned toward higher recall in the
"Investigate Quiet Risk" bucket (catching cases a human would otherwise never see) even though
this means more pages to review overall -- a team with tighter capacity should raise the
`rf_proba >= 0.5` threshold rather than skip that category, so the highest-risk quiet pages are
still seen first.

**Where it stops being valid:**
- **The decision point is fixed at 2026-04-30.** Scores reflect one snapshot; they do not
  update themselves as new daily data arrives. Re-scoring requires re-running the pipeline at
  a new decision point.
- **Only 13 of the queue's clients are truly out-of-sample.** The Random Forest was trained on
  38 clients and tested on the other 13. Scoring every page in `df` (including training-client
  pages, done above for full portfolio coverage) means predictions for those 38 clients are
  in-sample, not held-out -- treat their scores as somewhat more optimistic than the 13
  test-client scores.
- **This is decision support, not proof of cause.** A high score means "worth reviewing first,"
  not "guaranteed to decline" or "guaranteed to improve if refreshed." Only a controlled
  experiment (e.g. an A/B test on refreshed vs. untouched pages) could support a causal claim.
- **GSC/GA4 signals only.** No query-level, backlink, or competitor context -- rankings can move
  for reasons entirely outside this feature set (see Section 7 of the dataset guide: seasonality
  and consolidation can look like decline without being decline).

In [ ]:
# Quantify the in-sample vs. out-of-sample caveat named above
df['is_test_client'] = df['client_hash_id'].isin(test_clients)

print("Portfolio coverage by client group:")
print(df['is_test_client'].value_counts().rename({True: 'Test clients (true holdout)',
                                                     False: 'Train clients (in-sample)'}))
print(f"\n{df['is_test_client'].mean():.1%} of the full queue's rows come from truly held-out clients.")
print("The remaining rows use in-sample predictions -- flag this to any reviewer using the full queue.")

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on any page in this queue, a human should check:**
1. **Consolidation** -- did a sibling page on the same site absorb the lost traffic? (check
   related content/keyword groups)
2. **Seasonality** -- does this category naturally dip on the calendar? (compare to the same
   period in prior history where available)
3. **Minimum volume** -- is the page's traffic large enough that the pattern isn't just noise?
4. **For "Verify then Review" specifically** -- rule out a tracking artifact or a paid campaign
   before treating a spike as organic signal.

**No-go list -- never automate these:**
- Never auto-publish, auto-delete, or auto-redirect a page based on this score alone.
- Never treat FlyRank's own product flags (`health_score`, `needs_ctr_fix`, etc.) as ground
  truth if they're ever reintroduced here -- they're an output to compare against, never a
  model input (see `flyrank-context`).
- Never skip human sign-off for "Investigate Quiet Risk" pages -- by definition, no rule
  explains why the model flagged them, so a person must supply the reason before any action.
- Never publish or reconstruct raw URLs, client names, or query text anywhere downstream of
  this queue.

In [ ]:
forbidden_patterns = ['http://', 'https://', 'www.']
leak_check = action_queue.astype(str).apply(
    lambda col: col.str.contains('|'.join(forbidden_patterns), case=False, na=False)
).any()

print("Any column containing a raw URL-like string (should all be False):")
print(leak_check)

# Count how many "Investigate Quiet Risk" pages exist -- these ALWAYS require human sign-off
quiet_risk_count = (action_queue['action'] == 'Investigate Quiet Risk').sum()
print(f"\n'Investigate Quiet Risk' pages requiring mandatory human review: {quiet_risk_count:,}")

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Retrain or re-audit the pipeline if any of these show up in monthly monitoring:

| Signal | Trigger |
|---|---|
| **Precision@50 drop** | Measured monthly precision@50 falls meaningfully below the 0.98 validated here |
| **Base rate drift** | The declining-page share moves far from the ~56-58% seen in this snapshot (a sign the portfolio's behavior has changed) |
| **Client mix change** | New clients enter the portfolio with very different history depth or vertical -- the model has never seen their pattern |
| **Rule-trigger rate drift** | The share of pages with `any_rule_triggered == 1` drifts sharply from ~39% -- the baseline's own thresholds may need re-deriving from fresh percentiles |
| **Feature importance reshuffle** | Re-running feature importance shows a new feature dominating (>40%) that wasn't dominant before -- investigate for a new leakage source first |

In [ ]:
# Illustrative monitoring snapshot -- the numbers a future monthly check should compare against
monitoring_snapshot = {
    'precision_at_50': round(precision_at_k(test_results, 'rf_proba', 'is_declining', 50), 3),
    'base_rate': round(y_test.mean(), 3),
    'rule_trigger_rate': round(df['any_rule_triggered'].mean(), 3),
    'top_feature': importances.iloc[0]['feature'],
    'top_feature_share': round(importances.iloc[0]['importance'], 3),
    'decision_point': '2026-04-30',
}

for k, v in monitoring_snapshot.items():
    print(f"{k:22} {v}")

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to `work/outputs/` -- your paper builds on these files.*

Per the assignment's export rule: the queue CSV goes to `work/outputs/` and stays **out of
git by design** (the CI leak-guard blocks data files -- rerun this notebook to regenerate it
locally). Figures go to `work/figures/` and **are committed** -- the paper embeds them
directly. The metrics snapshot is saved as JSON (also committed) -- these numbers are the
receipts the paper's claims trace back to.

In [ ]:
import os, json
import matplotlib.pyplot as plt

os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

# --- Data export (gitignored, regenerated by re-running this notebook) ---
action_queue.to_csv('work/outputs/action_queue.csv', index=False)
print(f"Wrote work/outputs/action_queue.csv (gitignored) -- {len(action_queue):,} rows")

# --- Metrics export (committed -- the receipts) ---
with open('work/figures/monitoring_snapshot.json', 'w') as f:
    json.dump(monitoring_snapshot, f, indent=2)
print("Wrote work/figures/monitoring_snapshot.json (committed)")

# --- Figures export (committed) ---
fig, ax = plt.subplots(figsize=(6, 4))
df['action'].value_counts().plot(kind='barh', ax=ax)
ax.set_xlabel('Number of pages')
ax.set_title('Action playbook distribution')
plt.tight_layout()
plt.savefig('work/figures/action_distribution.png', dpi=150)
plt.close()
print("Wrote work/figures/action_distribution.png (committed)")

fig, ax = plt.subplots(figsize=(6, 4))
decay_table['decline_rate'].plot(kind='bar', ax=ax, color='#B84730')
ax.set_ylabel('Share declining')
ax.set_title('Decline rate by content age (decay/refresh insight)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('work/figures/decay_by_age.png', dpi=150)
plt.close()
print("Wrote work/figures/decay_by_age.png (committed)")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.